In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [3]:
df = pd.read_csv('spam.csv', encoding='latin-1')

In [4]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
df.drop(columns="Unnamed: 2",inplace=True, errors='ignore')
df.drop(columns="Unnamed: 3",inplace=True, errors='ignore')
df.drop(columns="Unnamed: 4",inplace=True, errors='ignore')

In [6]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.shape

(5572, 2)

In [8]:
df.isnull().sum()

v1    0
v2    0
dtype: int64

In [9]:
df.rename(columns={'v1':'label'}, inplace=True)
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [10]:
df = df.drop_duplicates()

In [11]:
df['label'] = df['label'].map({
    'spam':1,
    'ham':0
})

In [12]:
df.sample(15)

,label,v2
4391,0,what are your new years plans?
5113,0,U definitely need a module from e humanities d...
316,0,Hmmm... Guess we can go 4 kb n power yoga... H...
3676,0,Great! So what attracts you to the brothas?
1705,0,Yun ah.now Ì_ wkg where?btw if Ì_ go nus sc. Ì...
4437,0,Nothing will ever be easy. But don't be lookin...
4172,0,Ok... But they said i've got wisdom teeth hidd...
2584,0,Hi happy birthday. Hi hi hi hi hi hi hi
4942,0,Check mail.i have mailed varma and kept copy t...
3161,0,I can't describe how lucky you are that I'm ac...


In [13]:
df.columns = ['label','message']

In [14]:
df.sample(15)

,label,message
2691,0,Hey tmr meet at bugis 930 ?
1064,0,"That's fine, I'll bitch at you about it later ..."
3380,1,"complimentary 4 STAR Ibiza Holiday or å£10,000..."
4398,0,Yes just finished watching days of our lives. ...
2625,1,"FREE RING TONE just text \POLYS\"" to 87131. Th..."
2716,0,"House-Maid is the murderer, coz the man was mu..."
2530,0,So the sun is anti sleep medicine.
994,0,"I can't, I don't have her number!"
208,0,You please give us connection today itself bef...
4811,0,"fyi I'm at usf now, swing by the room whenever"


In [15]:
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from tqdm import tqdm

ps = PorterStemmer()

def transform_text(text):

    # Lowercase
    text = text.lower()

    # tokenize
    words = word_tokenize(text)

    # keep only letters and numbers
    words = [word for word in words if word.isalnum()]

    # remove stopwords
    words = [word for word in words if word not in stopwords.words('english')]

    # stemming
    words = [ps.stem(word) for word in words]

    return " ".join(words)

In [16]:
tqdm.pandas()
df['message'] = df['message'].progress_apply(transform_text)

100%|█████████████████████████████████████████████████████████████████████████████| 5169/5169 [00:22<00:00, 232.10it/s]


In [17]:
df.head()

,label,message
0,0,go jurong point crazi avail bugi n great world...
1,0,ok lar joke wif u oni
2,1,free entri 2 wkli comp win fa cup final tkt 21...
3,0,u dun say earli hor u c alreadi say
4,0,nah think goe usf live around though


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

In [19]:
X = tfidf.fit_transform(df['message']).toarray()

In [20]:
y = df['label']

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [22]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [23]:
y_pred = model.predict(X_test)

In [24]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9729206963249516

In [25]:
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,y_pred)

array([[888,   1],
       [ 27, 118]])

In [26]:
precision_score(y_test,y_pred)

0.9915966386554622

In [28]:
df.sample(10)

,label,message
4140,0,beauti truth express face could seen everyon d...
1291,0,hey babe saw came onlin second disappear happen
5148,0,k come wenev u lik come also tel vikki come ge...
744,0,men like shorter ladi gaze eye
5189,1,ree entri 2 weekli comp chanc win ipod txt pod...
1401,0,kaiez enjoy ur tuition gee thk e second option...
3860,1,free msg rington http wml 37819
3668,0,yeah imma come caus jay want drug
3206,0,phone weirdest auto correct
2511,0,er yep sure prop


In [30]:
sms = input("Enter Message: ")
processed = transform_text(sms)

vector = tfidf.transform([processed])
prediction = model.predict(vector)

if prediction[0] == 1:
    print("Spam")
else:   
    print("Ham")

Enter Message:  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


Spam


In [34]:
import pickle

pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(model, open('model.pkl', 'wb'))